In [ ]:
storage_account = "stretailbidevgk01"
container = "retail-data"

storage_key = dbutils.secrets.get(
    scope="kv-retail-bi-dev",
    key="adls-storage-key"
)

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    storage_key
)

print("ADLS authentication configured successfully")

In [ ]:
""" ****Weather landing to silver****"""
"""Weather landing-> Weather bronze-> Weather Silver->Data Quality Checks"""
     

display(dbutils.fs.ls("abfss://retail-data@stretailbidevgk01.dfs.core.windows.net/bronze/"))
display(dbutils.fs.ls("abfss://retail-data@stretailbidevgk01.dfs.core.windows.net/bronze/weather/"))


In [ ]:
"""Weather bronze file"""
weather_path = "abfss://retail-data@stretailbidevgk01.dfs.core.windows.net/bronze/weather/"
display(dbutils.fs.ls(weather_path))

In [ ]:
# Read Weather data from Bronze Delta
weather_path = (
    "abfss://retail-data@stretailbidevgk01.dfs.core.windows.net/bronze/weather"
)

df_weather_raw = spark.read.format("delta").load(weather_path)

print("Column count:", len(df_weather_raw.columns))
print("Columns:", df_weather_raw.columns)

display(df_weather_raw)

In [ ]:

# Bronze to Silver — Weather
from pyspark.sql.functions import col, to_date, explode, arrays_zip

# 1. Define Bronze and Silver paths
bronze_path = (
    "abfss://retail-data@stretailbidevgk01.dfs.core.windows.net/"
    "bronze/weather"
)

weather_silver_path = (
    "abfss://retail-data@stretailbidevgk01.dfs.core.windows.net/"
    "silver/weather"
)

# 2. Read from Bronze Delta
df_weather_bronze = (
    spark.read
    .format("delta")
    .load(bronze_path)
)

print("Bronze row count:", df_weather_bronze.count())
print("Bronze columns:", df_weather_bronze.columns)

# 3. Validate expected columns
required_columns = [
    "daily",
    "latitude",
    "longitude",
    "elevation",
    "timezone",
    "timezone_abbreviation",
    "utc_offset_seconds"
]

missing_columns = [
    c for c in required_columns
    if c not in df_weather_bronze.columns
]

if missing_columns:
    raise ValueError(
        f"Missing expected Bronze columns: {missing_columns}"
    )

# 4. Explode daily arrays and transform Bronze to Silver
df_weather_exploded = df_weather_bronze.select(
    col("latitude"),
    col("longitude"),
    col("elevation"),
    col("timezone"),
    col("timezone_abbreviation"),
    col("utc_offset_seconds"),
    explode(
        arrays_zip(
            col("daily.time"),
            col("daily.temperature_2m_max"),
            col("daily.temperature_2m_min"),
            col("daily.precipitation_sum"),
            col("daily.rain_sum"),
            col("daily.wind_speed_10m_max")
        )
    ).alias("weather")
)

df_weather_silver = (
    df_weather_exploded
    .select(
        to_date(col("weather.time")).alias("weather_date"),
        col("weather.temperature_2m_max").cast("double").alias("temperature_max_f"),
        col("weather.temperature_2m_min").cast("double").alias("temperature_min_f"),
        col("weather.precipitation_sum").cast("double").alias("precipitation_mm"),
        col("weather.rain_sum").cast("double").alias("rain_mm"),
        col("weather.wind_speed_10m_max").cast("double").alias("wind_speed_max_mph"),
        col("latitude").cast("double").alias("latitude"),
        col("longitude").cast("double").alias("longitude"),
        col("elevation").cast("double").alias("elevation"),
        col("timezone").alias("timezone"),
        col("timezone_abbreviation").alias("timezone_abbreviation"),
        col("utc_offset_seconds").cast("int").alias("utc_offset_seconds")
    )
    .filter(col("weather_date").isNotNull())
    .dropDuplicates(["weather_date"])
)

# 5. Run basic quality checks
print("Silver row count:", df_weather_silver.count())

"""Null dates"""
print(
    "Null weather dates:",
    df_weather_silver.filter(col("weather_date").isNull()).count()
)

print(
    "Invalid temperatures:",
    df_weather_silver.filter(
        col("temperature_min_f") > col("temperature_max_f")
    ).count()
)

print(
    "Negative precipitation:",
    df_weather_silver.filter(col("precipitation_mm") < 0).count()
)

print(
    "Negative rain:",
    df_weather_silver.filter(col("rain_mm") < 0).count()
)

print(
    "Negative wind speed:",
    df_weather_silver.filter(col("wind_speed_max_mph") < 0).count()
)

display(df_weather_silver)

# 6. Write Silver Delta
(
    df_weather_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(weather_silver_path)
)

# 7. Verify the persisted Silver data
df_weather_silver_check = (
    spark.read
    .format("delta")
    .load(weather_silver_path)
)

print("Verified Silver row count:", df_weather_silver_check.count())
display(df_weather_silver_check)
